In [15]:
import json
import re
from pathlib import Path


class LearningPathBuilder:
    def __init__(self, topics_path: str = "topics.json"):
        data = json.loads(Path(topics_path).read_text(encoding="utf-8"))
        self.topics = data["topics"]

    def _depth(self, topic_id: str, _cache: dict = {}) -> int:
        if topic_id in _cache:
            return _cache[topic_id]
        prereqs = self.topics[topic_id]["prerequisites"]
        depth = 0 if not prereqs else 1 + max(self._depth(p) for p in prereqs)
        _cache[topic_id] = depth
        return depth

    def _goal_topic(self, goal: str) -> str:
        matches = [tid for tid, info in self.topics.items() if goal in info["goals"]]
        if not matches:
            raise ValueError(f"No topics found for goal '{goal}'")
        return max(matches, key=self._depth)

    def _collect_needed(self, target: str, completed: set) -> set:
        needed = set()
        stack = [target]
        while stack:
            current = stack.pop()
            if current in completed or current in needed:
                continue
            needed.add(current)
            stack.extend(self.topics[current]["prerequisites"])
        return needed

    def _topological_sort(self, needed: set) -> list:
        visited = set()
        order = []

        def visit(topic_id):
            if topic_id in visited:
                return
            visited.add(topic_id)
            for prereq in self.topics[topic_id]["prerequisites"]:
                if prereq in needed:
                    visit(prereq)
            order.append(topic_id)

        for topic_id in needed:
            visit(topic_id)
        return order

    def build_path(self, completed: list, goal: str) -> dict:
        completed_set = set(completed)
        target = self._goal_topic(goal)

        if target in completed_set:
            return {"goal": goal, "target": target, "path": [], "message": "الهدف محقق بالفعل!"}

        needed = self._collect_needed(target, completed_set)
        ordered_ids = self._topological_sort(needed)

        path = []
        for i, topic_id in enumerate(ordered_ids, start=1):
            info = self.topics[topic_id]
            path.append({"step": i, "id": topic_id, "name": info["name"], "level": info["level"]})

        return {
            "goal": goal,
            "target": self.topics[target]["name"],
            "total_steps": len(path),
            "path": path,
        }


class NLPLearningAssistant:
    def __init__(self, topics_path: str = "topics.json"):
        data = json.loads(Path(topics_path).read_text(encoding="utf-8"))
        self.topics = data["topics"]
        self.path_builder = LearningPathBuilder(topics_path)

    def infer_completed_topics(self, text: str) -> dict:
        text_lower = text.lower()
        inferred = {}
        for topic_id, info in self.topics.items():
            for kw in info.get("keywords", []):
                if re.search(r"\b" + re.escape(kw) + r"\b", text_lower):
                    inferred[topic_id] = {"name": info["name"], "matched_keyword": kw}
                    break
        return inferred

    def analyze_and_recommend(self, text: str, goal: str, extra_completed: list = None) -> dict:
        inferred = self.infer_completed_topics(text)
        completed = set(inferred.keys())
        if extra_completed:
            completed.update(extra_completed)
        path_result = self.path_builder.build_path(completed=list(completed), goal=goal)
        return {
            "input_text": text,
            "inferred_topics": inferred,
            "combined_completed": sorted(completed),
            "goal": goal,
            "recommended_path": path_result,
        }


print("The classes were successfully defined.")

The classes were successfully defined.


In [16]:
result = assistant.analyze_and_recommend(
    text="I built my first ML prediction model.",
    goal="ai"
)

print("📝 The sentence you wrote:")
print(f'   "{result["input_text"]}"')
print()
print("🧠 The system understood that you completed:")
for topic_id, match in result["inferred_topics"].items():
    print(f"   ✓ {match['name']}")
print()
print(f"🎯 Goal: {result['goal']}")
print(f"🏁 Final Destination: {result['recommended_path']['target']}")
print()
print("🗺️  Recommended Path:")
for step in result["recommended_path"]["path"]:
    print(f"   {step['step']}. {step['name']}  ({step['level']})")

📝 The sentence you wrote:
   "I built my first ML prediction model."

🧠 The system understood that you completed:
   ✓ First ML Project (Regression/Classification)

🎯 Goal: ai
🏁 Final Destination: Computer Vision

🗺️  Recommended Path:
   1. Deep Learning Fundamentals  (intermediate)
   2. PyTorch / TensorFlow  (intermediate)
   3. Build a CNN Project  (intermediate)
   4. Computer Vision  (advanced)


In [1]:
import json
import re
from datetime import datetime
from pathlib import Path


class LearningPathBuilder:
    def __init__(self, topics_path="topics.json"):
        data = json.loads(Path(topics_path).read_text(encoding="utf-8"))
        self.topics = data["topics"]

    def _depth(self, topic_id, _cache={}):
        if topic_id in _cache:
            return _cache[topic_id]
        prereqs = self.topics[topic_id]["prerequisites"]
        depth = 0 if not prereqs else 1 + max(self._depth(p) for p in prereqs)
        _cache[topic_id] = depth
        return depth

    def _goal_topic(self, goal):
        matches = [tid for tid, info in self.topics.items() if goal in info["goals"]]
        if not matches:
            raise ValueError(f"No topics found for goal '{goal}'")
        return max(matches, key=self._depth)

    def _collect_needed(self, target, completed):
        needed = set()
        stack = [target]
        while stack:
            current = stack.pop()
            if current in completed or current in needed:
                continue
            needed.add(current)
            stack.extend(self.topics[current]["prerequisites"])
        return needed

    def _topological_sort(self, needed):
        visited = set()
        order = []

        def visit(topic_id):
            if topic_id in visited:
                return
            visited.add(topic_id)
            for prereq in self.topics[topic_id]["prerequisites"]:
                if prereq in needed:
                    visit(prereq)
            order.append(topic_id)

        for topic_id in needed:
            visit(topic_id)
        return order

    def build_path(self, completed, goal):
        completed_set = set(completed)
        target = self._goal_topic(goal)
        if target in completed_set:
            return {"goal": goal, "target": target, "path": [], "message": "الهدف محقق بالفعل!"}
        needed = self._collect_needed(target, completed_set)
        ordered_ids = self._topological_sort(needed)
        path = []
        for i, topic_id in enumerate(ordered_ids, start=1):
            info = self.topics[topic_id]
            path.append({"step": i, "id": topic_id, "name": info["name"], "level": info["level"]})
        return {"goal": goal, "target": self.topics[target]["name"], "total_steps": len(path), "path": path}


class NLPLearningAssistant:
    def __init__(self, topics_path="topics.json"):
        data = json.loads(Path(topics_path).read_text(encoding="utf-8"))
        self.topics = data["topics"]
        self.path_builder = LearningPathBuilder(topics_path)

    def infer_completed_topics(self, text):
        text_lower = text.lower()
        inferred = {}
        for topic_id, info in self.topics.items():
            for kw in info.get("keywords", []):
                if re.search(r"\b" + re.escape(kw) + r"\b", text_lower):
                    inferred[topic_id] = {"name": info["name"], "matched_keyword": kw}
                    break
        return inferred


class AILearningMentor:
    def __init__(self, topics_path="topics.json", progress_path="progress.json"):
        data = json.loads(Path(topics_path).read_text(encoding="utf-8"))
        self.topics = data["topics"]
        self.progress_path = Path(progress_path)
        self.path_builder = LearningPathBuilder(topics_path)
        self.nlp_assistant = NLPLearningAssistant(topics_path)
        self.progress = self._load_progress()

    def _load_progress(self):
        if self.progress_path.exists():
            return json.loads(self.progress_path.read_text(encoding="utf-8"))
        return {"goal": None, "completed": [], "history": []}

    def _save_progress(self):
        self.progress_path.write_text(json.dumps(self.progress, indent=2, ensure_ascii=False), encoding="utf-8")

    def set_goal(self, goal):
        self.progress["goal"] = goal
        self._save_progress()

    def add_completed(self, topic_id=None, text=None):
        newly_completed = []
        if topic_id:
            if topic_id not in self.topics:
                raise ValueError(f"Unknown topic id: {topic_id}")
            newly_completed.append(topic_id)
        if text:
            inferred = self.nlp_assistant.infer_completed_topics(text)
            newly_completed.extend(inferred.keys())
        added = []
        for tid in newly_completed:
            if tid not in self.progress["completed"]:
                self.progress["completed"].append(tid)
                self.progress["history"].append({"topic": tid, "name": self.topics[tid]["name"], "date": datetime.now().strftime("%Y-%m-%d")})
                added.append(tid)
        self._save_progress()
        return added

    def mentor_report(self):
        goal = self.progress.get("goal")
        completed = self.progress.get("completed", [])
        if not goal:
            return "لسا ما حددتي هدفك. استخدمي set_goal('ai' / 'robotics' / 'research') الأول."
        if not completed:
            return f"لسا ما سجّلتي أي إنجاز. استخدمي add_completed(...) لتسجيل أول topic خلصتيه نحو هدف '{goal}'."
        path_result = self.path_builder.build_path(completed=completed, goal=goal)
        lines = [f"📊 عندك {len(completed)} إنجاز مسجّل نحو هدف '{goal}':"]
        for entry in self.progress["history"]:
            lines.append(f"   ✓ {entry['name']}  ({entry['date']})")
        if not path_result["path"]:
            lines.append(f"\n🎉 مبروك! وصلتي للهدف: {path_result['target']}")
            return "\n".join(lines)
        next_step = path_result["path"][0]
        next_topic_info = self.topics[next_step["id"]]
        lines.append(f"\n🎯 خطوتك الجاية الموصى فيها: {next_step['name']}")
        lines.append(f"   (لأنها بتقربك من هدفك: {path_result['target']})")
        lines.append(f"\n💡 مشروع مقترح لهاي الخطوة:")
        lines.append(f"   {next_topic_info['project_idea']}")
        remaining = len(path_result["path"])
        lines.append(f"\n🗺️ باقي {remaining} خطوة/خطوات للوصول لهدفك الكامل.")
        return "\n".join(lines)


print("✅ كل الكلاسات جاهزة")

✅ كل الكلاسات جاهزة


In [2]:
mentor = AILearningMentor("topics.json", "progress.json")
mentor.set_goal("robotics")

mentor.add_completed(topic_id="python")
mentor.add_completed(topic_id="math")
mentor.add_completed(text="I finished statistics and did a data analysis project with pandas.")
mentor.add_completed(text="I built my first ML prediction model.")

print(mentor.mentor_report())

📊 عندك 5 إنجاز مسجّل نحو هدف 'robotics':
   ✓ Python Programming  (2026-09-07)
   ✓ Mathematics (Linear Algebra, Calculus)  (2026-09-07)
   ✓ Statistics & Probability  (2026-09-07)
   ✓ Data Analysis (Pandas, NumPy)  (2026-09-07)
   ✓ First ML Project (Regression/Classification)  (2026-09-07)

🎯 خطوتك الجاية الموصى فيها: Deep Learning Fundamentals
   (لأنها بتقربك من هدفك: Robotics: Control Systems & ROS)

💡 مشروع مقترح لهاي الخطوة:
   Build a simple feedforward neural network from scratch (no framework) on a toy dataset.

🗺️ باقي 5 خطوة/خطوات للوصول لهدفك الكامل.
